# Lab 3 — Napkin math, then a toy engine that proves it

**The claim you should be able to make when you finish:** *"Give me a model
config and a GPU and I will tell you, on paper, how much concurrency it holds
and how fast it decodes — and I have written the engine that shows the numbers
are right."*

This is the lab that pays off in every interview, because it turns "I have used
vLLM" into "I know what vLLM is doing and why". It is also the lab that fits a
notebook best: the napkin-math sheet *is* a notebook, and the toy engine runs
happily on a free T4 at GPT-2 scale.

Two halves:

* **Part A — the sheet.** Closed-form estimates for KV size, concurrency,
  decode speed, and cost. No GPU needed.
* **Part B — the engine.** ~300 lines: a paged block allocator, continuous
  batching, preemption. Measured against Part A's predictions on real GPT-2.

In [ ]:
# Cell 1 — idempotent bootstrap. No vLLM here: this lab needs only torch +
# transformers, both already present in Colab.
REPO   = "https://github.com/lsgrep/serv.git"
BRANCH = "main"

import os, subprocess, sys

if not os.path.isdir("serv"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
else:
    subprocess.run(["git", "-C", "serv", "pull", "--ff-only", "-q"], check=False)
sys.path.insert(0, os.path.abspath("serv"))

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "matplotlib", "pandas", "transformers"], check=True)

import servlab
env = servlab.notebook_setup()

# Part A — the napkin

## A1. KV cache per token

$$\text{bytes/token} = 2 \times L \times H_{kv} \times d_{head} \times \text{bytes/element}$$

* the **2** is K and V,
* $H_{kv}$ is **key/value** heads, not attention heads — GQA is the whole game,
* $d_{head}$ is often *not* `hidden / heads`; several configs set it explicitly.

Two numbers worth memorising, because they anchor everything else:

| model | KV per token (fp16) | why |
|---|---|---|
| GPT-2 | 36 KiB | 12 layers, no GQA |
| Llama-3.1-8B | 128 KiB | 32 layers, but 8 KV heads for 32 query heads |

Llama-3.1-8B has 2.7x the layers of GPT-2 and 65x the parameters, yet only 3.5x
the KV per token. That is GQA doing its job.

In [ ]:
from servlab import napkin as nk

print(f"{'model':<16}{'layers':>7}{'kv heads':>10}{'GQA':>6}{'KV/token':>12}{'@2k ctx':>12}")
for key in ["gpt2", "tinyllama-1.1b", "llama-3.2-1b", "llama-3.2-3b",
            "qwen2.5-3b", "qwen2.5-7b", "llama-3.1-8b"]:
    s = nk.MODELS[key]
    print(f"{key:<16}{s.n_layers:>7}{s.n_kv_heads:>10}{s.gqa_ratio:>5.0f}x"
          f"{nk.human_bytes(nk.kv_bytes_per_token(s)):>12}"
          f"{nk.human_bytes(nk.kv_bytes(s, 2048)):>12}")

In [ ]:
# What GQA is worth: the same models with multi-head attention instead.
from servlab.plots import use_style, SERIES
import matplotlib.pyplot as plt

use_style()
keys = ["llama-3.2-3b", "qwen2.5-3b", "qwen2.5-7b", "llama-3.1-8b"]
gqa  = [nk.kv_bytes_per_token(nk.MODELS[k]) / 1024 for k in keys]
mha  = [nk.kv_bytes_per_token(nk.ModelSpec(**{**nk.MODELS[k].__dict__,
                                              "n_kv_heads": nk.MODELS[k].n_heads})) / 1024
        for k in keys]

fig, ax = plt.subplots(figsize=(7.5, 4))
x = range(len(keys))
ax.bar([i - 0.19 for i in x], mha, width=0.36, color=SERIES[1], label="if it used MHA")
ax.bar([i + 0.19 for i in x], gqa, width=0.36, color=SERIES[0], label="actual (GQA)")
ax.set_xticks(list(x)); ax.set_xticklabels(keys, rotation=12)
ax.set_ylabel("KV per token (KiB)")
ax.set_title("grouped-query attention is a KV-cache optimisation")
ax.legend(loc="upper left")
plt.show()

## A2. From bytes to concurrency

VRAM is spent in a fixed order, and only the last slice is yours to trade:

```
total  =  weights  +  activations & CUDA context  +  KV cache  +  slack
```

Concurrency is `KV budget / (KV per token × context length)`. Two consequences
that catch people out:

* **Context length and concurrency are the same knob.** Halving `max_model_len`
  doubles the number of sequences you can hold. There is no third option.
* **Memory is a cliff, not a slope.** One model fits with 100 sequences of
  headroom; the next size up does not fit at all.

In [ ]:
for card in ["T4", "L4", "A100-40GB"]:
    print(nk.memory_report(card, "qwen2.5-3b", seq_len=2048))
    print()

In [ ]:
# Concurrency vs context length — the trade, drawn.
import matplotlib.pyplot as plt
from servlab.plots import SERIES

ctx = [512, 1024, 2048, 4096, 8192, 16384, 32768]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for i, card in enumerate(["T4", "L4", "A100-40GB"]):
    ys = [nk.max_concurrent_sequences(card, "qwen2.5-3b", seq_len=c) for c in ctx]
    ax.plot(ctx, [max(y, 0.1) for y in ys], marker="o", color=SERIES[i], label=card)
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xlabel("context length (tokens)"); ax.set_ylabel("concurrent sequences")
ax.set_title("Qwen2.5-3B fp16 — every doubling of context halves concurrency")
ax.legend(loc="upper right")
plt.show()

In [ ]:
# Quantising the KV cache is the cheapest concurrency you can buy.
print(f"{'kv dtype':<10}{'bytes/token':>14}{'seqs @ 2k on T4':>18}")
for dt in ["fp16", "fp8", "int4"]:
    print(f"{dt:<10}{nk.human_bytes(nk.kv_bytes_per_token('qwen2.5-3b', dt)):>14}"
          f"{nk.max_concurrent_sequences('T4', 'qwen2.5-3b', 2048, kv_dtype=dt):>18,.0f}")
print("\n(FP8 KV needs Ada or newer — on a T4 this row is arithmetic, not an option.)")

## A3. Why decode is slow and batching is nearly free

A decode step reads **all the weights** to produce **one token per sequence**.
Arithmetic intensity — FLOPs per byte moved — is therefore about `2 × batch`,
while a T4 needs ~200 FLOPs/byte to saturate its ALUs.

At batch 1 you are using roughly 1% of the card's compute. The GPU is idle,
waiting on memory. That single fact explains:

* why continuous batching is worth so much,
* why a bigger batch costs almost no extra time until the KV term grows,
* why decode gets *slower* per token as context grows (the KV read grows),
* why prefill and decode have completely different bottlenecks and should be
  measured separately.

In [ ]:
import matplotlib.pyplot as plt
from servlab.plots import SERIES, STATUS

SPEC = nk.MODELS["qwen2.5-3b"]
batches = [1, 2, 4, 8, 16, 32, 64, 128]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for i, ctx_len in enumerate([256, 2048, 8192]):
    axes[0].plot(batches, [nk.decode_tokens_per_s("T4", SPEC, b, ctx_len) for b in batches],
                 marker="o", color=SERIES[i], label=f"ctx {ctx_len}")
    axes[1].plot(batches, [nk.arithmetic_intensity(SPEC, b, ctx_len) for b in batches],
                 marker="o", color=SERIES[i], label=f"ctx {ctx_len}")
axes[0].set_xscale("log", base=2); axes[0].set_xlabel("batch size")
axes[0].set_ylabel("output tokens/s"); axes[0].set_title("T4 decode throughput")
axes[0].legend(loc="upper left")
axes[1].axhline(nk.ridge_point("T4"), color=STATUS["critical"], linestyle=":", linewidth=1.5)
axes[1].annotate("T4 ridge point — compute bound above this line",
                 xy=(1, nk.ridge_point("T4")), xytext=(0, 6), textcoords="offset points",
                 color=STATUS["critical"], fontsize=9)
axes[1].set_xscale("log", base=2); axes[1].set_yscale("log")
axes[1].set_xlabel("batch size"); axes[1].set_ylabel("FLOPs per byte")
axes[1].set_title("arithmetic intensity — decode never gets there")
axes[1].legend(loc="lower right")
plt.tight_layout(); plt.show()

# Part B — the toy engine

Everything in Part A is a claim. Now measure it, on a model small enough that a
free T4 runs the whole thing in seconds.

## B1. The KV cache, by subtraction

Time each decode step twice: once carrying `past_key_values`, once recomputing
the whole prefix every step. The second curve bends upward — step *n* re-attends
over *n* tokens — while the first stays roughly flat.

The gap between the curves is exactly what the KV cache buys, and this plot is
the clearest single artifact in the whole ladder.

In [ ]:
from servlab.toy.engine import load, decode_curve, spec_from_hf, measured_kv_bytes

model, tok = load("gpt2")
spec = spec_from_hf(model, "gpt2")
print(f"{spec.name}: {spec.n_layers} layers, {spec.n_heads} heads, "
      f"head_dim {spec.head_dim}, {spec.params/1e6:.0f}M params")
print(f"predicted KV/token: {nk.human_bytes(nk.kv_bytes_per_token(spec, next(model.parameters()).element_size()))}")

In [ ]:
# ~1 minute on a T4; a few minutes on CPU (drop n_new to 32 if so).
rows = (decode_curve(model, tok, n_new=96, use_cache=True) +
        decode_curve(model, tok, n_new=96, use_cache=False))

from servlab.plots import decode_curves
decode_curves(rows, title="GPT-2: time per token, with and without a KV cache");

In [ ]:
# Quantify it rather than eyeballing the chart.
with_c = [r["ms"] for r in rows if r["cache"].startswith("with")]
no_c   = [r["ms"] for r in rows if r["cache"].startswith("no")]
print(f"with cache:  first {with_c[0]:6.1f} ms   last {with_c[-1]:6.1f} ms   "
      f"growth {with_c[-1]/with_c[0]:.2f}x")
print(f"no cache:    first {no_c[0]:6.1f} ms   last {no_c[-1]:6.1f} ms   "
      f"growth {no_c[-1]/no_c[0]:.2f}x")
print(f"\nspeedup at the last token: {no_c[-1]/with_c[-1]:.1f}x")
print("\nGPT-2 has a 1024-token limit, so the quadratic term stays modest here.")
print("Extrapolate to a 32k context and the no-cache curve is off the chart —")
print("that is why no production stack has ever shipped without a KV cache.")

In [ ]:
# Does the cache weigh what Part A said it would? This is the check that turns
# the formula from something you memorised into something you trust.
import torch

ids = tok("the cache is a tensor and it has a size", return_tensors="pt").input_ids.to(model.device)
with torch.inference_mode():
    out = model(ids, use_cache=True)

measured = measured_kv_bytes(out.past_key_values)
per_token = nk.kv_bytes_per_token(spec, next(model.parameters()).element_size())
predicted = per_token * ids.shape[-1]
print(f"context length     {ids.shape[-1]} tokens")
print(f"predicted KV       {nk.human_bytes(predicted)}")
print(f"measured KV        {nk.human_bytes(measured)}")
print(f"ratio              {measured/predicted:.3f}")

## B2. Paged KV allocation

Before PagedAttention, a server reserved a contiguous slab of
`max_model_len` per sequence. A request that generated 20 tokens with a 2048
limit held 99% of its reservation empty. Under those rules a card that could
physically hold 200 sequences would serve 12.

Paging hands out fixed-size blocks (16 tokens in vLLM) on demand, so waste is
bounded by **one partly-filled block per sequence** — a few percent instead of
90-something. Nothing else about the model changes; it is pure bookkeeping, and
it is most of the throughput gap between a 2022 server and a 2024 one.

In [ ]:
from servlab.toy import BlockAllocator

BLOCK = 16
alloc = BlockAllocator(num_blocks=64, block_size=BLOCK)

lengths = {"a": 20, "b": 33, "c": 5}
for sid, n in lengths.items():
    alloc.allocate(sid, n)
    print(f"{sid}: {n:>3} tokens -> {len(alloc.blocks_for(sid))} blocks "
          f"({alloc.slots_left(sid, n)} slots idle in the last one)")

print(f"\npool: {alloc.num_used}/{alloc.num_blocks} blocks used ({alloc.usage:.0%})")
print(f"internal fragmentation: {alloc.fragmentation(lengths):.1%}")

MAX_LEN = 2048
reserved = sum(MAX_LEN for _ in lengths)
print(f"\ncontiguous reservation would have used {reserved:,} token slots "
      f"for {sum(lengths.values())} tokens -> {1 - sum(lengths.values())/reserved:.1%} waste")

## B3. Continuous batching and preemption

Static batching runs a batch to completion before starting the next, so one
500-token generation holds eight 20-token ones hostage. Continuous batching
admits a new request the step after a slot frees.

The scheduler loop is four lines, and everything a production scheduler adds —
chunked prefill, priority, prefix reuse, swapping — is a refinement of them:

```
every step:
    admit from the waiting queue while there are seats and blocks
    append one token to every running sequence
    if a sequence needs a block and none is free -> preempt someone
    retire finished sequences and free their blocks
```

Preemption is the interesting one. When the pool is exhausted the engine evicts
a *running* sequence — the newest, which has invested the least work — and puts
it back in the queue. In `recompute` mode its KV cache is thrown away and its
prefill happens again later. No request fails. Throughput just quietly goes to
waste, which is why preemption count is the metric that tells you a server is in
trouble before latency does.

In [ ]:
from servlab.toy import Scheduler, SchedulerConfig, Request
from servlab.toy.engine import BatchedEngine

# Deliberately tiny KV pool: 24 MiB, so preemption happens in a lab-sized run.
engine = BatchedEngine(model, tok, kv_budget_bytes=24 * 1024**2, block_size=16,
                       config=SchedulerConfig(max_num_seqs=8, watermark=0.02))

prompts = [
    "The history of computing began",
    "In a serving system, the key value cache",
    "Grouped query attention reduces",
    "The scheduler admits a request when",
    "Memory bandwidth limits decode because",
    "A preemption occurs when the block pool",
    "Continuous batching differs from static batching in that",
    "The arithmetic intensity of a decode step is",
    "Paged attention was introduced to solve",
    "Time to first token is dominated by",
    "Throughput and goodput diverge when",
    "The ridge point of a GPU tells you",
]
for i, p in enumerate(prompts):
    engine.submit(p, max_tokens=48, req_id=f"r{i}")

rows = engine.run()
print(f"\n{len(rows)} steps, {len(engine.scheduler.finished)} requests finished, "
      f"{engine.scheduler.total_preemptions} preemptions")

In [ ]:
# The same dashboard function that plotted a live vLLM in lab 1 — because the
# toy engine reports the same metric names. That is not a coincidence; it is the
# point. If you can read this chart you can read the real one.
from servlab.plots import dashboard
dashboard(rows, title="toy engine: continuous batching on GPT-2");

In [ ]:
print(engine.text("r0")[:300])

## B4. Turn the knobs and watch the shape change

This is where a notebook beats a terminal: three lines, rerun, see the effect.
Try each of these and predict the outcome *before* running it:

* `max_num_seqs=2` — fewer seats. Queue grows, latency rises, preemption stops.
* `kv_budget_bytes=8 * 1024**2` — a starved pool. Preemption storm; watch
  throughput fall while nothing errors.
* `watermark=0.0` — no admission reserve. The engine admits a request and
  immediately evicts someone to make room. Thrash, not progress. This one is the
  best demonstration of why admission control exists.
* `block_size=64` — coarser pages. Fragmentation up, bookkeeping down.

In [ ]:
import matplotlib.pyplot as plt
from servlab.plots import use_style, SERIES

use_style()
variants = {
    "baseline":         dict(kv_budget_bytes=24 * 1024**2, config=SchedulerConfig(max_num_seqs=8, watermark=0.02)),
    "starved pool":     dict(kv_budget_bytes=8 * 1024**2,  config=SchedulerConfig(max_num_seqs=8, watermark=0.02)),
    "no watermark":     dict(kv_budget_bytes=8 * 1024**2,  config=SchedulerConfig(max_num_seqs=8, watermark=0.0)),
    "few seats":        dict(kv_budget_bytes=24 * 1024**2, config=SchedulerConfig(max_num_seqs=2, watermark=0.02)),
}

results = {}
for name, kw in variants.items():
    e = BatchedEngine(model, tok, block_size=16, **kw)
    for i, p in enumerate(prompts):
        e.submit(p, max_tokens=48, req_id=f"r{i}")
    r = e.run()
    results[name] = {"steps": len(r), "preemptions": e.scheduler.total_preemptions,
                     "finished": len(e.scheduler.finished), "rows": r}
    print(f"{name:<16} {len(r):>5} steps  {e.scheduler.total_preemptions:>4} preemptions  "
          f"{len(e.scheduler.finished)}/{len(prompts)} finished")

fig, ax = plt.subplots(figsize=(7.5, 4.2))
for i, (name, r) in enumerate(results.items()):
    ax.plot([x["t"] for x in r["rows"]], [x["kv_usage"] for x in r["rows"]],
            color=SERIES[i], label=name)
ax.set_xlabel("seconds"); ax.set_ylabel("KV cache (%)")
ax.set_title("the same 12 requests under four scheduler configurations")
ax.legend(loc="lower right")
plt.show()

## What to be able to say afterwards

1. **The KV formula, from memory,** and what each term does. Especially why it
   is KV heads and not attention heads.
2. **Why decode is memory bound and prefill is compute bound**, and what follows
   from that about batching.
3. **What paging fixed** — reservation waste, not compute — and what it costs
   (a block table, a custom attention kernel).
4. **What a preemption is, and why it is invisible in a latency dashboard**
   until it has already cost you throughput.
5. **How you would size a deployment**: pick context, compute KV per token,
   subtract weights, divide. Two minutes on a whiteboard.

**Next:** lab 4 moves from serving to training memory, where the same
predict-then-measure loop meets an OOM.